# 00 — Setup

Creates the Drive workspace, installs packages, and writes `config.py`.

**This notebook does not touch the data.** Run it before reading `PROTOCOL.md` if you like;
run `01` only after the protocol is signed.

Run order: `00 → 01 → 02 → 03 → 04 → 05`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, sys, json, platform, subprocess

DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'

SUBDIRS = [
    'data/raw_original',
    'data/raw_improved',
    'data/interim',
    'results',
    'figures',
    'checkpoints',
    'src',
]

for d in SUBDIRS:
    os.makedirs(os.path.join(DRIVE_ROOT, d), exist_ok=True)

print('workspace ready at', DRIVE_ROOT)
for d in SUBDIRS:
    print('  ', d)

workspace ready at /content/drive/MyDrive/research/ids-label-correction
   data/raw_original
   data/raw_improved
   data/interim
   results
   figures
   checkpoints
   src


In [3]:
!pip -q install xgboost scikit-learn pandas numpy scipy matplotlib tqdm
print('packages installed')

packages installed


In [4]:
CONFIG_PY = r'''
"""Frozen configuration for the CICIDS2017 label-correction study.

Nothing in this file may change after PROTOCOL.md is signed. If it must change,
record the change and the reason in PROTOCOL_AMENDMENTS.md.
"""

import os

DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'

RAW_ORIGINAL = os.path.join(DRIVE_ROOT, 'data/raw_original')
RAW_IMPROVED = os.path.join(DRIVE_ROOT, 'data/raw_improved')
INTERIM      = os.path.join(DRIVE_ROOT, 'data/interim')
RESULTS      = os.path.join(DRIVE_ROOT, 'results')
FIGURES      = os.path.join(DRIVE_ROOT, 'figures')
CHECKPOINTS  = os.path.join(DRIVE_ROOT, 'checkpoints')

# --- frozen experimental constants -----------------------------------------
SEEDS = [11, 23, 37, 51, 73]

SPLIT_PROTOCOLS = ['random_70_30', 'day_ordered']

# how flows labelled "Attempted" in the improved dataset are treated
ATTEMPTED_POLICIES = ['as_attack', 'as_benign', 'dropped']

MODELS = ['logreg', 'random_forest', 'xgboost']

# equivalence margin for H4 (TOST)
EQUIV_MARGIN = 0.02

# gate G2: minimum acceptable matched-flow rate for Arm B
MIN_MATCH_RATE = 0.20

# gate G4: maximum tolerated cross-session drift
REPRO_TOLERANCE = 0.02

# how the improved release's "Infiltration - Portscan" flows are counted.
# 'Infiltration' (default) or 'PortScan'. This is a reported decision.
INFIL_PORTSCAN_AS = 'Infiltration'

# rows carrying no label at all are dropped and the count is reported
DROP_NULL_LABELS = True

# --- day-ordered split ------------------------------------------------------
TRAIN_DAYS = ['monday', 'tuesday', 'wednesday']
TEST_DAYS  = ['thursday', 'friday']

# --- Kaggle slugs -----------------------------------------------------------
# VERIFIED slug for the corrected data (Liu et al., IEEE CNS 2022):
KAGGLE_IMPROVED = 'ernie55ernie/improved-cicids2017-and-csecicids2018'

# The original CICIDS2017 has several Kaggle mirrors and they are NOT equivalent.
# You must use a mirror carrying GeneratedLabelledFlows / TrafficLabelling_,
# which retains Source IP / Destination IP / Flow ID. Mirrors carrying only
# MachineLearningCVE will make Arm B impossible. Notebook 01 helps you check.
KAGGLE_ORIGINAL = ''   # <-- fill in during notebook 01, then never change

# --- column harmonisation ---------------------------------------------------
# The two versions name the same fields differently. Keys are the canonical
# names used throughout the pipeline.
CANONICAL = {
    'src_ip':    ['source ip', 'src ip', 'srcip'],
    'src_port':  ['source port', 'src port', 'srcport'],
    'dst_ip':    ['destination ip', 'dst ip', 'dstip'],
    'dst_port':  ['destination port', 'dst port', 'dstport'],
    'protocol':  ['protocol'],
    'timestamp': ['timestamp'],
    'label':     ['label'],
    'flow_id':   ['flow id', 'flowid', 'id'],
    'attempted': ['attempted category'],
}

# columns never used as model features
NON_FEATURES = ['src_ip', 'dst_ip', 'timestamp', 'label', 'flow_id',
                'attempted', 'day', 'version']
'''

import os
cfg_path = os.path.join(DRIVE_ROOT, 'src', 'config.py')
with open(cfg_path, 'w') as f:
    f.write(CONFIG_PY)
print('wrote', cfg_path)

wrote /content/drive/MyDrive/research/ids-label-correction/src/config.py


In [5]:
HELPERS_PY = r'''
"""Shared helpers. Imported by every notebook so the logic lives in one place."""

import os, re, hashlib
import numpy as np
import pandas as pd

import config as C


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def norm_col(c):
    """Lowercase, collapse whitespace, strip. Both CIC releases have stray spaces."""
    return re.sub(r'\s+', ' ', str(c)).strip().lower()


def harmonise(df):
    """Rename columns to canonical names where a mapping exists; normalise the rest."""
    df = df.copy()
    df.columns = [norm_col(c) for c in df.columns]
    rename = {}
    for canon, variants in C.CANONICAL.items():
        for v in variants:
            if v in df.columns and canon not in rename.values():
                rename[v] = canon
                break
    return df.rename(columns=rename)


def feature_columns(df):
    return [c for c in df.columns if c not in C.NON_FEATURES]


def clean_features(df, report=None):
    """Protocol section 6, steps 3-4. Returns cleaned df and a report dict."""
    rep = {} if report is None else report
    feats = feature_columns(df)
    X = df[feats].replace([np.inf, -np.inf], np.nan)
    before = len(df)
    keep = ~X.isna().any(axis=1)
    rep['rows_before'] = before
    rep['rows_dropped_nan'] = int((~keep).sum())
    df = df.loc[keep].copy()
    X = X.loc[keep]
    const = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
    rep['constant_columns'] = const
    df = df.drop(columns=const)
    rep['rows_after'] = len(df)
    return df, rep


def dedup(df, report=None):
    """Protocol section 6, step 5. Duplicate removal is reported, never silent."""
    rep = {} if report is None else report
    feats = feature_columns(df)
    n_before = len(df)
    df2 = df.drop_duplicates(subset=feats + ['label'])
    rep['exact_duplicates_removed'] = int(n_before - len(df2))
    rep['duplicate_rate'] = float((n_before - len(df2)) / max(n_before, 1))
    return df2, rep


def binarise(labels):
    """BENIGN -> 0, everything else -> 1. Null labels -> -1 (must be handled upstream)."""
    s = labels.astype(str).str.strip().str.upper()
    out = (s != 'BENIGN').astype(int)
    out[labels.isna() | (s == 'NAN')] = -1
    return out


def is_attempted(labels):
    """True where the improved release marks the flow as an unsuccessful attempt.

    The 'attempted category' column in the improved CSVs uses a sentinel for
    non-attempted flows rather than a null, so it cannot be tested with notna().
    The label string is the reliable signal.
    """
    return labels.astype(str).str.contains('attempted', case=False, na=False)


# Ordered rules. Order is load-bearing and every choice here is a documented
# decision, not an accident:
#   - 'web attack' before 'brute force', or "Web Attack - Brute Force" would
#     be swallowed by the brute-force rule
#   - 'infiltration' before 'portscan', so the improved release's
#     "Infiltration - Portscan" (71,767 flows) counts as Infiltration, which is
#     the attack it belongs to. Set C.INFIL_PORTSCAN_AS = 'PortScan' to invert
#     this; the choice materially changes the PortScan and Infiltration rows of
#     the class census, so state it in the paper either way.
#   - 'ddos' before 'dos'
FAMILY_RULES = [
    ('heartbleed', 'Heartbleed'),
    ('web attack', 'WebAttack'),
    ('xss', 'WebAttack'),
    ('sql', 'WebAttack'),
    ('infiltration', '__INFIL__'),
    ('portscan', 'PortScan'),
    ('port scan', 'PortScan'),
    ('ddos', 'DDoS'),
    ('dos', 'DoS'),
    ('patator', 'BruteForce'),
    ('brute', 'BruteForce'),
    ('bot', 'Bot'),
]


def coarse_class(labels):
    """Collapse the raw labels into families for per-class reporting.

    Null labels map to 'NULL_LABEL' — never silently into an 'Other' bucket.
    """
    infil_portscan = getattr(C, 'INFIL_PORTSCAN_AS', 'Infiltration')
    s = labels.astype(str).str.strip().str.lower()
    out = []
    for raw, v in zip(labels, s):
        if pd.isna(raw) or v in ('nan', '', 'none'):
            out.append('NULL_LABEL')
            continue
        if v.startswith('benign'):
            out.append('BENIGN')
            continue
        hit = 'Other'
        for needle, fam in FAMILY_RULES:
            if needle in v:
                if fam == '__INFIL__':
                    hit = infil_portscan if ('portscan' in v or 'port scan' in v) \
                          else 'Infiltration'
                else:
                    hit = fam
                break
        out.append(hit)
    return pd.Series(out, index=labels.index)


def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > y) for x in a for y in b)
    lt = sum((x < y) for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))


def save_table(df, name):
    path = os.path.join(C.RESULTS, name)
    df.to_csv(path, index=False)
    print('saved', path, df.shape)
    return path
'''

helpers_path = os.path.join(DRIVE_ROOT, 'src', 'helpers.py')
with open(helpers_path, 'w') as f:
    f.write(HELPERS_PY)
print('wrote', helpers_path)

wrote /content/drive/MyDrive/research/ids-label-correction/src/helpers.py


In [6]:
# record the environment — gate G4 needs this later
import platform, json, os
import importlib

env = {'python': platform.python_version()}
for mod in ['numpy', 'pandas', 'sklearn', 'xgboost', 'scipy']:
    try:
        env[mod] = importlib.import_module(mod).__version__
    except Exception as e:
        env[mod] = f'ERROR {e}'

path = os.path.join(DRIVE_ROOT, 'results', 'environment.json')
with open(path, 'w') as f:
    json.dump(env, f, indent=2)
print(json.dumps(env, indent=2))
print('\nsaved', path)

{
  "python": "3.12.13",
  "numpy": "2.0.2",
  "pandas": "2.2.3",
  "sklearn": "1.6.1",
  "xgboost": "3.4.0",
  "scipy": "1.16.3"
}

saved /content/drive/MyDrive/research/ids-label-correction/results/environment.json


## Done

Next: read and sign `PROTOCOL.md`, then run `01_download.ipynb`.

Do not skip the signing step. It is the thing that makes the eventual paper defensible.